# 16 — `LearnableGain`: auto-recover spatial gain drift across a long scan

Long overnight operando scans accumulate a smooth **spatial gain drift** on
the area detector — thermal expansion in the readout electronics,
sensor-temperature variation, and slow scintillator degradation. pyFAI does
not model this, so the drift leaks into the integrated I(Q) and shows up as
fake intensity changes in the operando trace.

`LearnableGain` is an `nn.Module` whose per-pixel gain field is a torch
parameter. Trained jointly with the integration kernel, it absorbs the drift
into a multiplicative correction so the integrated profile stays clean.

This notebook is **self-contained** — it plants a known gain field on a
synthetic frame and recovers it inline with the public API, on CPU. No C
binaries, no external data.


In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import torch
from scipy.ndimage import gaussian_filter

from midas_integrate.params import IntegrationParams
from midas_integrate_v2 import (
    LearnableGain, gain_smoothness_prior, gain_unity_prior,
    integrate_with_corrections, spec_from_v1_params,
)

torch.set_default_dtype(torch.float64)


DIPlib -- a quantitative image analysis library
Version 3.5.2 (Dec 27 2024)
For more information see https://diplib.org


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Synthesize a ring frame and a planted gain field

The clean frame is a stack of six concentric Debye–Scherrer rings with a
little Poisson-ish read noise. The planted gain `gain_truth` is a 5%
Gaussian-smooth random field (σ = 20 px) — exactly the kind of slow spatial
drift we want to recover.


In [2]:
NY = NZ = 256
rng = np.random.default_rng(0)

BC_y, BC_z = NY / 2.0, NZ / 2.0
Y, Z = np.meshgrid(np.arange(NY), np.arange(NZ), indexing="xy")
R = np.sqrt((Y - BC_y) ** 2 + (Z - BC_z) ** 2)

img = np.zeros((NZ, NY))
for r0 in np.linspace(20, 110, 6):
    img += 1000.0 * np.exp(-((R - r0) / 1.4) ** 2)
img += rng.normal(0.0, 5.0, size=img.shape)
img = np.clip(img, 0, None)

raw = rng.normal(0.0, 1.0, size=(NZ, NY))
smoothed = gaussian_filter(raw, sigma=20.0)
smoothed /= np.max(np.abs(smoothed)) + 1e-30
gain_truth = 1.0 + 0.05 * smoothed          # ~+/-5% smooth drift

img_drifted = img * gain_truth
print(f"frame {img.shape}, planted gain range "
      f"[{gain_truth.min():.3f}, {gain_truth.max():.3f}]")


frame (256, 256), planted gain range [0.950, 1.034]


## 2. Integration spec

`spec_from_v1_params` turns a v1 `IntegrationParams` into the differentiable
v2 spec consumed by `integrate_with_corrections`. The **clean-frame**
integration is our training target; the gain module has to make the drifted
frame integrate to the same profile.


In [3]:
p = IntegrationParams(
    NrPixelsY=NY, NrPixelsZ=NZ,
    pxY=200.0, pxZ=200.0, Lsd=1_000_000.0,
    BC_y=BC_y, BC_z=BC_z, RhoD=120.0,
    RMin=10.0, RMax=120.0, RBinSize=1.0,
    EtaMin=-180.0, EtaMax=180.0, EtaBinSize=10.0,
)
spec = spec_from_v1_params(p, requires_grad=False)

img_t = torch.as_tensor(img, dtype=torch.float64)
img_drift_t = torch.as_tensor(img_drifted, dtype=torch.float64)
target = integrate_with_corrections(img_t, spec).detach()
print("target profile shape:", tuple(target.shape))


target profile shape: (36, 110)


## 3. Train `LearnableGain` to reverse the drift

We divide the drifted frame by the current gain estimate, integrate, and
minimise the mismatch against the clean target. Two priors keep the solution
well-posed:

- `gain_unity_prior` — pulls the mean gain toward 1 (fixes the global scale
  gauge that integration alone can't pin down),
- `gain_smoothness_prior` — penalises high spatial frequency, matching the
  smooth physical drift.


In [4]:
gain = LearnableGain(NrPixelsZ=NZ, NrPixelsY=NY, scale=0.1)
optim = torch.optim.Adam(gain.parameters(), lr=0.02)

for step in range(40):
    optim.zero_grad()
    adjusted = img_drift_t / gain().clamp(min=1e-6)
    out = integrate_with_corrections(adjusted, spec)
    loss = (out - target).pow(2).mean()
    loss = (loss
            + 1e-4 * gain_unity_prior(gain)
            + 1e-3 * gain_smoothness_prior(gain))
    loss.backward()
    optim.step()
    if step % 10 == 0:
        print(f"  step {step:3d}  loss={float(loss):.6e}")

gain_recovered = gain().detach().numpy()


/var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/ipykernel_43768/2117675871.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print(f"  step {step:3d}  loss={float(loss):.6e}")


  step   0  loss=1.216652e+03


  step  10  loss=7.904724e+01


  step  20  loss=5.755425e+01


  step  30  loss=1.033032e+01


## 4. How close did we get?

In [5]:
rmse = float(np.sqrt(np.mean((gain_recovered - gain_truth) ** 2)))
amp = float(gain_truth.max() - gain_truth.min())
print(f"recovered-vs-truth RMSE : {rmse:.4e}")
print(f"planted drift amplitude : {amp:.4e}")
print(f"RMSE as %% of amplitude : {100 * rmse / amp:.2f} %%")
assert rmse < 0.05, "gain recovery did not converge"


recovered-vs-truth RMSE : 1.0629e-02
planted drift amplitude : 8.4419e-02
RMSE as %% of amplitude : 12.59 %%


## 5. Why this matters

- The truth gain is a 5% Gaussian-smooth random field (σ = 20 px); the
  learned field tracks it to a small fraction of the planted amplitude.
- The learned gain is applied as a **multiplicative per-pixel correction** to
  every subsequent frame — the integration kernel itself never changes.
- This is the operando overnight-scan story: without correction, a 1% gain
  drift would masquerade as a 1% intensity change in the time-resolved
  profile. pyFAI / calib2 calibrate geometry but not a per-pixel gain field.

To turn the recovered field into figures or a report, the
`dev/paper/runners/run_learnable_gain_demo.py` script wraps this exact logic
and writes `gain_recovered.npy`, `gain_truth.npy`, and a `REPORT.md` under
`dev/paper/runs/learnable_gain_demo/`.
